In [1]:
%pip install geopandas --quiet

StatementMeta(, 72dafc53-aeea-446a-8085-fd7894df22bf, 7, Finished, Available, Finished, True)


[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: /nfs4/pyenv-5ac72faf-bd8c-4351-b5bb-aaa1cb13b78a/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.



In [2]:
import zipfile, glob
import pandas as pd
import geopandas as gpd
from pyspark.sql import functions as F

WS = "ws-urban-analytics"
BRONZE = f"abfss://{WS}@onelake.dfs.fabric.microsoft.com/lh_bronze.Lakehouse"
def read_bronze_table(name):
    for path in (f"{BRONZE}/Tables/dbo/{name}", f"{BRONZE}/Tables/{name}"):
        try:
            return spark.read.format("delta").load(path)
        except Exception:
            continue
    raise ValueError(f"Bronze table not found: {name}")
def save(pdf, table):
    spark.createDataFrame(pdf).write.mode("overwrite") \
         .option("overwriteSchema", "true").saveAsTable(table)
    print(f"{table}: {len(pdf):,} rows")

StatementMeta(, 72dafc53-aeea-446a-8085-fd7894df22bf, 9, Finished, Available, Finished, False)

In [3]:
notebookutils.fs.cp(f"{BRONZE}/Files/raw/taxi_zones/taxi_zones.zip", "file:/tmp/taxi_zones.zip")
with zipfile.ZipFile("/tmp/taxi_zones.zip") as z:
    z.extractall("/tmp/taxi_zones")
shp = glob.glob("/tmp/taxi_zones/**/*.shp", recursive=True)[0]
zones_proj = gpd.read_file(shp).dissolve(by="LocationID", as_index=False)
centroids = zones_proj.geometry.centroid.to_crs(epsg=4326)
zones_wgs = zones_proj.to_crs(epsg=4326)[["LocationID", "geometry"]]
geo = pd.DataFrame({
    "location_id": zones_proj["LocationID"].astype(int),
    "centroid_lat": centroids.y,
    "centroid_lon": centroids.x,
})
lookup = (spark.read.option("header", True)
          .csv(f"{BRONZE}/Files/raw/taxi_zones/taxi_zone_lookup.csv").toPandas())
lookup = lookup.rename(columns={"LocationID": "location_id", "Borough": "borough",
                                "Zone": "zone", "service_zone": "service_zone"})
lookup["location_id"] = lookup["location_id"].astype(int)
silver_zones = lookup.merge(geo, on="location_id", how="left")
save(silver_zones, "silver_zones")

StatementMeta(, 72dafc53-aeea-446a-8085-fd7894df22bf, 10, Finished, Available, Finished, False)

silver_zones: 265 rows


In [4]:
air = read_bronze_table("bronze_openaq").toPandas()
air["date"] = pd.to_datetime(air["date_utc"], utc=True).dt.tz_convert("America/New_York").dt.date
air["units"] = air["units"].str.lower()
factor = pd.Series(1.0, index=air.index)
factor[(air["parameter"] == "no2") & (air["units"] == "ppm")] = 1000.0
factor[(air["parameter"] == "no2") & (air["units"].str.contains("g/m"))] = 1 / 1.88
for col in ["value_avg", "value_min", "value_max"]:
    air[col] = air[col] * factor
air["unit"] = air["parameter"].map({"pm25": "µg/m³", "no2": "ppb"})
air = air[air["value_avg"].notna() & (air["value_avg"] >= 0)]
air = air[air["coverage_pct"].isna() | (air["coverage_pct"] >= 50)]
air = air.drop_duplicates(subset=["sensor_id", "date"])
stations = air[["location_id", "latitude", "longitude"]].drop_duplicates("location_id")
pts = gpd.GeoDataFrame(stations, geometry=gpd.points_from_xy(stations.longitude, stations.latitude), crs=4326)
joined = gpd.sjoin(pts, zones_wgs, how="left", predicate="within")[["location_id", "LocationID"]]
joined = joined.rename(columns={"LocationID": "zone_id"}).drop_duplicates("location_id")
air = air.merge(joined, on="location_id", how="left")
air["in_nyc_zone"] = air["zone_id"].notna()
air["zone_id"] = air["zone_id"].astype("float")
silver_air = air[["date", "location_id", "location_name", "sensor_id", "parameter",
                  "value_avg", "value_min", "value_max", "unit", "coverage_pct",
                  "latitude", "longitude", "zone_id", "in_nyc_zone"]]
spark.createDataFrame(silver_air) \
     .withColumn("zone_id", F.col("zone_id").cast("int")) \
     .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_air_quality")
print(f"silver_air_quality: {len(silver_air):,} rows, "
      f"stations in NYC zones: {joined['zone_id'].notna().sum()} / {len(joined)}")

StatementMeta(, 72dafc53-aeea-446a-8085-fd7894df22bf, 11, Finished, Available, Finished, False)

silver_air_quality: 2,002 rows, stations in NYC zones: 15 / 24


In [5]:
fx = read_bronze_table("bronze_fx").toPandas().dropna()
fx["date"] = pd.to_datetime(fx["date"])
fx = fx.sort_values("date").drop_duplicates("date").set_index("date")
full_range = pd.date_range(fx.index.min(), pd.Timestamp.today().normalize(), freq="D")
fx_daily = fx.reindex(full_range)
fx_daily["is_ecb_fixing_day"] = fx_daily["usd_per_eur"].notna()
fx_daily["usd_per_eur"] = fx_daily["usd_per_eur"].ffill()
fx_daily["eur_per_usd"] = 1 / fx_daily["usd_per_eur"]
fx_daily = fx_daily.reset_index(names="date")
fx_daily["date"] = fx_daily["date"].dt.date
save(fx_daily[["date", "usd_per_eur", "eur_per_usd", "is_ecb_fixing_day"]], "silver_fx_daily")

StatementMeta(, 72dafc53-aeea-446a-8085-fd7894df22bf, 12, Finished, Available, Finished, False)

silver_fx_daily: 4,345 rows


In [6]:
gdp = read_bronze_table("bronze_gdp").toPandas()
gdp = gdp.dropna(subset=["gdp_usd"]).drop_duplicates(["country_iso3", "year"])
gdp["year"] = gdp["year"].astype(int)
save(gdp[["country_iso3", "year", "gdp_usd"]].sort_values("year"), "silver_gdp")

StatementMeta(, 72dafc53-aeea-446a-8085-fd7894df22bf, 13, Finished, Available, Finished, False)

silver_gdp: 66 rows
